In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
import spacy
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device allocated for ablation operations: {device}\n")

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

OUTPUT_DIR = "/kaggle/working"
OUTPUT_EXCEL_PATH = os.path.join(OUTPUT_DIR, "step3_word_ablation_pos_analysis.xlsx")
OUTPUT_CSV_PATH = os.path.join(OUTPUT_DIR, "step3_all_word_ablations.csv")

MAX_WORDS_PER_TWEET = 35
ABLATION_BATCH_SIZE = 128 

BASE_INPUT_PATH = "/kaggle/input/datasets/claudiacornacchia/embedding/"
csv_path = os.path.join(BASE_INPUT_PATH, "base_map.csv")
pt_path = os.path.join(BASE_INPUT_PATH, "embeddings_1024d.pt")

if not os.path.exists(csv_path) or not os.path.exists(pt_path):
    raise FileNotFoundError(f"Required files missing in {BASE_INPUT_PATH}. Check dataset links.")

print(f"Loading Base Map: {csv_path}")
print(f"Loading Pre-computed Tensor: {pt_path}")

df = pd.read_csv(csv_path, dtype={"tweet_id": str})
df = df[df["text"].notna()].copy().reset_index(drop=True)

original_embeddings = torch.load(pt_path, map_location=device).float()
original_embeddings = F.normalize(original_embeddings, p=2, dim=1)

if len(df) != len(original_embeddings):
    raise ValueError(f"Shape mismatch: CSV has {len(df)} rows, but PT tensor has {len(original_embeddings)} vectors.")

print(f"Successfully mounted whole dataset framework: {len(df)} entries.")

emb_model_id = "Qwen/Qwen3-Embedding-0.6B"
print(f"Mounting {emb_model_id} for variant evaluations...")
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id, trust_remote_code=True)
emb_tokenizer.padding_side = "right"
emb_model = AutoModel.from_pretrained(emb_model_id, trust_remote_code=True).to(device)
emb_model.eval()

BRAND_TERMS = {"united", "usairways", "us", "airways", "american", "americanair", "southwest", "southwestair", "jetblue", "jetblues", "virgin", "virginamerica", "delta"}
OPERATIONAL_LEMMAS = {"cancel", "delay", "lose", "miss", "wait", "rebook", "refund", "change", "board", "fly", "arrive", "depart", "land", "book", "reschedule", "check", "connect", "respond", "answer", "pay", "charge", "flight", "luggage", "baggage", "bag", "gate", "seat", "ticket", "reservation", "connection", "crew", "airport", "cancelled", "canceled", "delayed", "boarding"}
NEGATION_TERMS = {"not", "no", "never", "n't", "cannot", "cant", "can't", "won't", "dont", "don't", "didn't", "isn't", "wasn't", "nothing", "none", "neither", "nor"}
COURTESY_TERMS = {"thanks", "thank", "please", "sorry", "hi", "hello", "help", "appreciate", "kindly", "great", "thx"}
NOISE_TERMS = {"http", "https", "www", "url", "urlword", "mentionword", "uh", "uhm", "um", "lol", "jk", "amp", "dm"}

def remove_word_by_span(text, start, end):
    modified = text[:start] + " " + text[end:]
    return re.sub(r"\s+", " ", modified).strip() if modified else "empty"

def classify_word(word):
    text = word.text.lower()
    lemma = word.lemma_.lower()
    pos = word.pos_
    if text.startswith("@"): return "MENTION"
    if text.startswith("http") or "www." in text: return "URL"
    if text in BRAND_TERMS or lemma in BRAND_TERMS: return "BRAND"
    if pos == "PROPN": return "PROPER_NOUN"
    if lemma in OPERATIONAL_LEMMAS or text in OPERATIONAL_LEMMAS: return "OPERATIONAL_TERM"
    if text in NEGATION_TERMS or lemma in NEGATION_TERMS: return "NEGATION"
    if text in COURTESY_TERMS or lemma in COURTESY_TERMS: return "COURTESY_CUSTOMER_CARE"
    if text in NOISE_TERMS or lemma in NOISE_TERMS: return "NOISE_PLATFORM_WORD"
    if pos in {"VERB", "AUX"}: return "VERB_AUX"
    if pos == "NOUN": return "COMMON_NOUN"
    if pos == "ADJ": return "ADJECTIVE"
    if pos == "ADV": return "ADVERB"
    if pos in {"DET", "ADP", "CCONJ", "SCONJ", "PART", "PRON"}: return "FUNCTION_WORD"
    if pos == "NUM": return "NUMBER"
    return "OTHER"

def is_valid_ablation_word(word):
    return not (word.is_space or word.is_punct or len(word.text.strip()) == 0)

ablation_records = []
current_batch_texts = []
current_batch_meta = []

print("\nExecuting Leave-One-WORD-Out (LOWO) Stress-Test...")
pbar = tqdm(total=len(df), desc="Streaming Dataset Ablations")

for idx, row in df.iterrows():
    text = str(row["text"])
    doc = nlp(text)
    
    tweet_baseline_vector = original_embeddings[idx].unsqueeze(0)
    
    valid_words = [t for t in doc if is_valid_ablation_word(t)][:MAX_WORDS_PER_TWEET]
    
    for word_pos, word in enumerate(valid_words):
        ablated_text = remove_word_by_span(text, word.idx, word.idx + len(word.text))
        
        meta = {
            "tweet_id": row.get("tweet_id", idx),
            "airline": row.get("airline", "Unknown"),
            "sentiment": row.get("airline_sentiment", row.get("sentiment", "Unknown")),
            "topic_id": row.get("topic_id", -1),
            "original_text": text,
            "ablated_text": ablated_text,
            "word": word.text,
            "lemma": word.lemma_,
            "pos": word.pos_,
            "tag": word.tag_,
            "is_stop": word.is_stop,
            "word_category": classify_word(word),
            "word_position": word_pos,
            "baseline_vector": tweet_baseline_vector  
        }
        
        current_batch_texts.append(ablated_text)
        current_batch_meta.append(meta)
        
        if len(current_batch_texts) == ABLATION_BATCH_SIZE:
            inputs = emb_tokenizer(current_batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            with torch.no_grad():
                outputs = emb_model(**inputs)
                mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                sum_emb = torch.sum(outputs.last_hidden_state * mask, 1)
                sum_msk = torch.clamp(inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float().sum(1), min=1e-9)
                ablated_vectors = F.normalize(sum_emb / sum_msk, p=2, dim=1)
                
            for i, metadata in enumerate(current_batch_meta):
                orig_v = metadata["baseline_vector"]
                abl_v = ablated_vectors[i].unsqueeze(0)
                sim = F.cosine_similarity(orig_v, abl_v).item()
                
                del metadata["baseline_vector"]  
                metadata["cosine_similarity"] = sim
                metadata["impact"] = 1.0 - sim
                ablation_records.append(metadata)
                
            current_batch_texts, current_batch_meta = [], []
            
    pbar.update(1)
pbar.close()

if current_batch_texts:
    inputs = emb_tokenizer(current_batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = emb_model(**inputs)
        mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_emb = torch.sum(outputs.last_hidden_state * mask, 1)
        sum_msk = torch.clamp(inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float().sum(1), min=1e-9)
        ablated_vectors = F.normalize(sum_emb / sum_msk, p=2, dim=1)
    for i, metadata in enumerate(current_batch_meta):
        sim = F.cosine_similarity(metadata["baseline_vector"], ablated_vectors[i].unsqueeze(0)).item()
        metadata["cosine_similarity"] = sim
        metadata["impact"] = 1.0 - sim
        del metadata["baseline_vector"]
        ablation_records.append(metadata)

ablation_df = pd.DataFrame(ablation_records)

category_summary = ablation_df.groupby("word_category").agg(
    n_words=("impact", "size"), mean_impact=("impact", "mean"),
    median_impact=("impact", "median"), std_impact=("impact", "std"), max_impact=("impact", "max")
).reset_index().sort_values("mean_impact", ascending=False)

pos_summary = ablation_df.groupby("pos").agg(
    n_words=("impact", "size"), mean_impact=("impact", "mean"),
    median_impact=("impact", "median"), std_impact=("impact", "std"), max_impact=("impact", "max")
).reset_index().sort_values("mean_impact", ascending=False)

word_summary = ablation_df.groupby(["word", "lemma", "pos", "word_category"]).agg(
    n_occurrences=("impact", "size"), mean_impact=("impact", "mean"),
    median_impact=("impact", "median"), max_impact=("impact", "max")
).reset_index().sort_values(["mean_impact", "n_occurrences"], ascending=[False, False])

word_summary_min5 = word_summary[word_summary["n_occurrences"] >= 5].copy()

print("\n" + "=" * 80)
print("WHOLE POPULATION ABLATION AND POS SUMMARY ANALYSIS")
print("=" * 80)
print("\n CATEGORY SUMMARY MATRIX:")
print(category_summary.to_string(index=False))
print("\n POS SUMMARY MATRIX:")
print(pos_summary.to_string(index=False))
print("\n" + "=" * 80)

ablation_df.to_csv(OUTPUT_CSV_PATH, index=False)
try:
    with pd.ExcelWriter(OUTPUT_EXCEL_PATH, engine="openpyxl") as writer:
        category_summary.to_excel(writer, sheet_name="category_summary", index=False)
        pos_summary.to_excel(writer, sheet_name="pos_summary", index=False)
        word_summary_min5.to_excel(writer, sheet_name="word_summary_min5", index=False)
        ablation_df.to_excel(writer, sheet_name="all_word_ablations", index=False)
    print(f"\n Definitive Full-Scale Validation Excel saved to: {OUTPUT_EXCEL_PATH}")
except Exception as e:
    print(f"\n Serialization Warning: {e}")
